# 02 — Verbalized Uncertainty

LLMs can be asked to express their confidence in words — but this rarely matches actual calibration.

## 'I Don't Know' Calibration

When an LLM says 'I'm confident that...' or 'I'm not sure, but...' — does this correlate with whether it's correct?

**Verbalized confidence** is when the model expresses uncertainty in words. Studies show:
- LLMs are overconfident: they say 'I'm confident' even when wrong at a high rate
- Confidence expressions correlate with answer correctness less than calibrated probabilities do
- More capable models are better calibrated in their verbalizations
- RLHF training can make models more or less calibrated depending on the reward signal

**Expected Calibration Error (ECE)**: measures the gap between confidence and accuracy:
$$ECE = \sum_{m=1}^{M} \frac{|B_m|}{n} |\text{acc}(B_m) - \text{conf}(B_m)|$$
where samples are grouped into M bins by confidence level.

**Reliability diagrams**: plot accuracy vs confidence. A perfectly calibrated model falls on the diagonal. Over-confidence shows as points above the diagonal; underconfidence below.

In [ ]:
# Calibration of verbalized confidence scores
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500

# Simulate LLM answers with verbalized confidence
# 'confidence' is what the model claims; 'correct' is ground truth
confidence_levels = np.random.choice([0.1, 0.3, 0.5, 0.7, 0.9], size=n,
                                      p=[0.1, 0.15, 0.2, 0.3, 0.25])

# Overconfident model: actual accuracy is lower than stated confidence
def simulate_overconfident(conf):
    # Actual accuracy = conf * 0.7 + bias toward overconfidence
    true_acc = conf * 0.6 + 0.15
    return np.random.binomial(1, np.clip(true_acc, 0, 1))

correct = np.array([simulate_overconfident(c) for c in confidence_levels])

def compute_ece(confidences, corrects, n_bins=5):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0
    bins = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        acc = corrects[mask].mean()
        conf = confidences[mask].mean()
        ece += mask.mean() * abs(acc - conf)
        bins.append((conf, acc, mask.sum()))
    return ece, bins

ece, bins = compute_ece(confidence_levels, correct)
print(f'ECE (overconfident model): {ece:.4f}')

# Reliability diagram
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

confs = [b[0] for b in bins]
accs = [b[1] for b in bins]
counts = [b[2] for b in bins]

ax1.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax1.bar(confs, accs, width=0.15, alpha=0.5, label='Actual accuracy', color='steelblue')
ax1.bar(confs, confs, width=0.15, alpha=0.3, label='Stated confidence', color='tomato')
ax1.set_xlabel('Confidence')
ax1.set_ylabel('Accuracy')
ax1.set_title(f'Reliability Diagram (ECE={ece:.3f})')
ax1.legend()

# Compare calibrated vs uncalibrated
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

# Temperature scaling calibration
def temperature_scale(logits, T):
    return 1 / (1 + np.exp(-logits / T))

logits = np.log(confidence_levels / (1 - confidence_levels + 1e-8))

# Find best temperature on the data
best_T, best_ece = 1.0, 1.0
for T in np.arange(0.5, 5.0, 0.1):
    scaled = temperature_scale(logits, T)
    e, _ = compute_ece(scaled, correct)
    if e < best_ece:
        best_ece = e
        best_T = T

calibrated = temperature_scale(logits, best_T)
cal_ece, cal_bins = compute_ece(calibrated, correct)

ax2.plot([0, 1], [0, 1], 'k--')
ax2.scatter([b[0] for b in bins], [b[1] for b in bins], label=f'Original (ECE={ece:.3f})', color='tomato', s=80)
ax2.scatter([b[0] for b in cal_bins], [b[1] for b in cal_bins], label=f'Temp-scaled T={best_T:.1f} (ECE={cal_ece:.3f})', color='steelblue', s=80)
ax2.set_xlabel('Confidence'); ax2.set_ylabel('Accuracy')
ax2.set_title('Temperature Scaling Calibration')
ax2.legend()

plt.tight_layout()
plt.savefig('/tmp/calibration.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Best temperature: {best_T:.1f}, ECE improved from {ece:.3f} to {cal_ece:.3f}')